# Logistic regression-based cache eviction policy

### Does a logistic regression model provide powerful cache eviction decisions?

In this notebook we implement a pipeline (data preparation, data generation, validation, training, and testing) for a **Logistic Regression** model as we did for the LSTM, using the same datasets and evaluation metrics.

*What is our goal?*
We aim at comparing a complex neural network against a simpler baseline to determine if a linear-based model is sufficient for cache eviction decisions.

*What do we expect?*
While Logistic Regression is computationally efficient, it is likely too simplistic for this task. It lacks the capacity to capture complex access patterns or temporal dependencies. Consequently, we expect it to underperform compared to the LSTM, which is specifically designed to learn midterm and long-term dependencies within sequence data.

---

### Imports

In [ ]:
from dataclasses import dataclass

import joblib
import numpy as np
import pandas as pd
from box import Box
from joblib import parallel_backend
from ray.util.joblib import register_ray
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GridSearchCV,
    train_test_split,
)
from sklearn.preprocessing import OneHotEncoder

from components.const import (
    DATASET_PROCESSED_FEATURE_COLUMNS,
    MODEL_LR_TRAINED_DYNAMIC_FILE_PATH,
    MODEL_LR_TRAINED_STATIC_FILE_PATH,
    MODEL_METRICS_ACCURACY_NAME,
    MODEL_METRICS_CLASS_REPORT_NAME,
    MODEL_METRICS_COHEN_KAPPA_SCORE_NAME,
    MODEL_METRICS_MACRO_AVG_NAME,
    MODEL_METRICS_WEIGHTED_AVG_NAME,
    TENSOR_FEATURES_DIM,
)
from components.dataset.columns.setter import set_dataset_column
from components.dataset.io.loader import load_dataset
from components.dataset.io.locator import get_dataset_abs_path
from components.evaluation.model.metrics.calculator import (
    calculate_model_metrics,
)
from components.validation.time_series_cv.builder import (
    build_time_series_split,
)
from const import (
    DATA_STATIC_MODE,
    DATASET_COLUMN_LR_PREVIOUS_REQUEST_PREFIX_NAME,
    DATASET_COLUMN_REQUEST_NAME,
)
from pipeline.config.configurator import prepare_pipeline_config
from pipeline.const import DATASET_PROCESSED_TYPE, DATASET_RESET_INDEX_DROP
from pipeline.steps.data_preparer import prepare_data
from pipeline.steps.data_preprocessor import preprocess_data
from pipeline.steps.simulator import run_simulations

---

### Configurations

In [ ]:
# Load pipeline configuration
pipeline_config = prepare_pipeline_config()

# Define configuration for Logistic Regression
lr_config = Box(
    {
        "encoder": {
            "handle_unknown": "ignore",  # Encoder's strategy to handle unknown classes
        },
        "model": {
            "class_weight": "balanced",  # Class weight to apply to the model
            "seq_len": 25,  # Sequence length of past requests to pass to the model as features
        },
        "validation": {
            "num_folds": 5,  # Number of folds for time series split
            "scoring": "f1_weighted",  # Metric to evaluate models during validation
            "search_space": {
                "C": [
                    0.5,
                    1.0,
                ],  # Inverse of regularization strength
                "max_iter": [
                    300,
                ],  # Max number of iterations for the solver's convergence
                "penalty": ["l2"],  # Regularization type
                "solver": ["lbfgs"],  # Optimizer algorithm
            },
        },
    },
)

---

### 1. Data Preparation and Data Preprocessing

These steps aim to generate and explore the dataset, as well as process it by removing missing values and building relevant features. These two steps are the same as those used in the LSTM pipeline. To provide the Logistic Regression model with information about previously accessed keys, the last `seq_len` accesses are extracted as separate categorical features and one-hot encoded. This allows the model to capture patterns explicitly. Finally, the dataset is split into training and testing sets.

In [ ]:
# Data preparation and preprocessing
prepare_data()
preprocess_data()

# Configuration data
data_mode = pipeline_config.data.general.mode
min_key = pipeline_config.data.general.keys.min
train_size = pipeline_config.dataset.splits.training
shuffle = pipeline_config.data_loader.validation.shuffle
seq_len = lr_config.model.seq_len
encoder_handle_unknown = lr_config.encoder.handle_unknown

# Load processed dataset
df = load_dataset(
    get_dataset_abs_path(
        DATASET_PROCESSED_TYPE,
        data_mode,
    ),
)

# Create shifted previous-request columns
prev_requests = []
for i in range(1, seq_len + 1):
    col_name = f"{DATASET_COLUMN_LR_PREVIOUS_REQUEST_PREFIX_NAME}{i}"
    set_dataset_column(
        df,
        col_name,
        df[DATASET_COLUMN_REQUEST_NAME].shift(i).astype(str),
    )
    prev_requests.append(col_name)

# Drop rows with incomplete history
df = df.iloc[seq_len:].reset_index(drop=DATASET_RESET_INDEX_DROP)

# Split train and test sets
df_train, df_test = train_test_split(
    df,
    train_size=train_size,
    shuffle=shuffle,
)

# One-hot encode previous requests
encoder = OneHotEncoder(handle_unknown=encoder_handle_unknown)
encoder.fit(df_train[prev_requests])

# Features as the combination of
# already existing ones and encodings
X_train = np.concatenate(
    [
        df_train[DATASET_PROCESSED_FEATURE_COLUMNS].values,
        encoder.transform(df_train[prev_requests]).toarray(),
    ],
    axis=TENSOR_FEATURES_DIM,
)
X_test = np.concatenate(
    [
        df_test[DATASET_PROCESSED_FEATURE_COLUMNS].values,
        encoder.transform(df_test[prev_requests]).toarray(),
    ],
    axis=TENSOR_FEATURES_DIM,
)

# Targets
y_train = df_train[DATASET_COLUMN_REQUEST_NAME].values.astype(int) - min_key
y_test = df_test[DATASET_COLUMN_REQUEST_NAME].values.astype(int) - min_key

---

### 2. Validation and Training

These steps find and train the best logistic regression model. The best hyperparameters are found by performing a grid search cross-validation with time series split to preserve temporal order. The validation strategy is almost identical to those used in the LSTM pipeline. The best logistic regression model is re-fitted over the whole training set with the best hyperparameters found and saved for further usage.

In [ ]:
# Define a class to wrap Logistic Regression model to
@dataclass
class LogisticRegressionWrapper:
    """This class wraps the logistic regression model.

    Attributes:
        model: The logistic regression model to wrap.
        encoder: One-hot encoder used for keys.
        seq_len: Sequence length of embedded keys.
    """

    model: LogisticRegression
    encoder: OneHotEncoder
    seq_len: int


# Configuration data
num_folds = lr_config.validation.num_folds
class_weight = lr_config.model.class_weight
search_space = lr_config.validation.search_space
scoring = lr_config.validation.scoring
n_jobs = pipeline_config.resources.general.num_cpus

# Define Time Series Split object
tscv = build_time_series_split(num_folds)

# Define a Logistic Regression model
lr = LogisticRegression(class_weight=class_weight)

# Define Grid Search Cross-Validation object
grid_search = GridSearchCV(
    estimator=lr,
    param_grid=search_space,
    cv=tscv,
    scoring=scoring,
)

# Run grid search in a distributed way with Ray
register_ray()
with parallel_backend("ray", n_jobs=n_jobs):
    grid_search.fit(X_train, y_train)

# Determine the path to save model to
if data_mode == DATA_STATIC_MODE:
    model_save_path = MODEL_LR_TRAINED_STATIC_FILE_PATH
else:
    model_save_path = MODEL_LR_TRAINED_DYNAMIC_FILE_PATH

# Wrap model
model = LogisticRegressionWrapper(
    grid_search.best_estimator_,
    encoder,
    seq_len,
)

# Save model
joblib.dump(model, model_save_path)

### 3. Testing

This step aims at evaluating the best logistic regression model using the same metrics as those used for LSTM.

In [ ]:
# Use the best model for making predictions
# over the testing set
y_pred = model.model.predict(X_test)

# Calculate evaluation metrics
metrics = calculate_model_metrics(
    y_test,
    y_pred,
)

# Prepare metrics
class_report = metrics[MODEL_METRICS_CLASS_REPORT_NAME]
accuracy = class_report[MODEL_METRICS_ACCURACY_NAME]
cohen_kappa_score = metrics[MODEL_METRICS_COHEN_KAPPA_SCORE_NAME]
metrics_df = pd.DataFrame(class_report).T
class_wise_metrics = metrics_df.drop(
    [
        MODEL_METRICS_ACCURACY_NAME,
        MODEL_METRICS_MACRO_AVG_NAME,
        MODEL_METRICS_WEIGHTED_AVG_NAME,
    ],
)
aggregated_metrics = metrics_df.loc[
    [MODEL_METRICS_MACRO_AVG_NAME, MODEL_METRICS_WEIGHTED_AVG_NAME],
    :,
]

# Show metrics in tabular form
print("Class-wise metrics:" + "\n" + "-" * 60 + "\n", class_wise_metrics)
print("\nAggregated metrics:" + "\n" + "-" * 60 + "\n", aggregated_metrics)
print("\nAccuracy:" + "\n" + "-" * 60 + "\n", accuracy)
print("\nCohen Kappa score:" + "\n" + "-" * 60 + "\n", cohen_kappa_score)

### 4. Simulations

In this final step, we simulate several eviction policies as in the LSTM pipeline, with the addition of a Logistic Regression-based eviction policy. This policy exploits the pre-trained model to decide which key to evict from the cache when it is full, based on predicted request probabilities.

In [ ]:
run_simulations()